# MicroBrainNet-DA — Single-Technique Baseline: CDAN
### PlantVillage (source) -> PlantDoc (target) | TinyML / ESP32-S3 | INT8 < 500 KB

---

## What this notebook is

This is one of **seven** baseline-comparison notebooks (DANN, CDAN, DSAN, CORAL, IM, MMD, MCC).
Every notebook in the set shares the **exact same** data pipeline, architecture, teacher,
knowledge distillation, pseudo-labeling and training schedule (Parts 0–2, 4–12 below are
byte-identical text across all seven). The **only** thing that changes between notebooks is one
config line in Part 0:

```python
DA_METHOD = "cdan"
```


---
# Part 0: Imports & Configuration

In [ ]:
# ============================================================
# PART 0: IMPORTS AND CONFIGURATION
# ============================================================
import os, sys, time, random, warnings, copy, json, glob, math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter
from collections import Counter, OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as tvm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, classification_report, roc_curve, auc)
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

SEED = 42

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ------------------------------------------------------------------
# SINGLE-TECHNIQUE BASELINE SWITCH
# This is the ONLY line that differs between the DANN / CDAN / DSAN / CORAL / IM / MMD / MCC
# comparison notebooks. Architecture (Part 2), data pipeline (Part 1), teacher (Part 5), KD,
# pseudo-labeling, schedules and every hyperparameter besides the technique-specific lambda
# stay identical across all seven notebooks, so any accuracy delta between them is attributable
# to the domain-adaptation technique itself.
# ------------------------------------------------------------------
DA_METHOD = "cdan"     # one of: coral, mmd, im, mcc, dann, cdan, dsan
assert DA_METHOD in {"coral", "mmd", "im", "mcc", "dann", "cdan", "dsan"}, \
    f"Unknown DA_METHOD '{DA_METHOD}'"

OUTPUT_DIR  = f"/kaggle/working/microbrainnet_da_{DA_METHOD}"
FIGURES_DIR = f"{OUTPUT_DIR}/figures"
MODELS_DIR  = f"{OUTPUT_DIR}/models"
TABLES_DIR  = f"{OUTPUT_DIR}/tables"
for d in [FIGURES_DIR, MODELS_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

# ------------------------------------------------------------------
# Student sized for a <500 KB INT8 TinyML budget. 128px + wider channels lands around 230 KB
# INT8 -- comfortably inside budget, ~250-400 ms/inference on an ESP32-S3 with PSRAM.
# ------------------------------------------------------------------
IMG_SIZE          = 128
TEACHER_IMG_SIZE  = 224
NUM_CLASSES       = 3
STUDENT_CHANNELS  = (32, 48, 64, 96, 96)

BATCH_SIZE        = 48    # source
TARGET_BATCH_SIZE = 32    # smaller batch + drop_last=False keeps all 364 target images
TEACHER_BATCH_SIZE= 24

CLASSES = ["Tomato_Septoria_Leaf_Spot", "Potato_Early_Blight", "Corn_Leaf_Blight"]

# ---------------- stage switches ----------------
RUN_SANITY_GATE        = True    # Part 0B: source-only run. DO NOT SKIP on the first execution.
USE_BG_RANDOMIZATION   = True
USE_TEACHER_KD         = True
TEACHER_USE_PSEUDO     = False   # OFF by default -- establish the clean number first
STUDENT_USE_PSEUDO     = True    # gated + auto-disabling (see Part 3/4)
RUN_MULTI_SEED         = True    # Part 7 (expensive: re-runs the student per seed) -- turned ON:
                                  # a single-seed number is not defensible on a 79-image test set
SEEDS                  = [42, 1337, 2024]

# ---------------- teacher ----------------
TEACHER_BACKBONE   = 'mobilenet_v3_large'
TEACHER_EPOCHS     = 80
TEACHER_LR_BACKBONE= 3e-4        # discriminative LR instead of a cosmetic block freeze
TEACHER_LR_HEAD    = 3e-3
TEACHER_WARMUP     = 3
TEACHER_PATIENCE   = 30

# ---------------- student ----------------
EPOCHS         = 120
LR             = 1.5e-3
WEIGHT_DECAY   = 1e-4
LABEL_SMOOTHING= 0.1
WARMUP_EPOCHS  = 5
PATIENCE       = 40

# ---------------- domain adaptation: per-technique lambdas ----------------
# Only ONE of these is ever active per notebook (selected by DA_METHOD above), but all seven
# are defined here so the loss-function cell and train_da_epoch are identical text across every
# notebook -- nothing to keep in sync by hand.
LAMBDA_CORAL_MAX  = 0.5
LAMBDA_MMD_MAX    = 0.5
LAMBDA_IM_MAX     = 0.5
LAMBDA_MCC_MAX    = 1.0
LAMBDA_DANN_MAX   = 1.0    # adversarial domain-classifier loss weight (Ganin et al., 2016)
LAMBDA_CDAN_MAX   = 1.0    # conditional adversarial loss weight (Long et al., 2018)
LAMBDA_DSAN_MAX   = 0.5    # local/subdomain MMD weight (Zhu et al., 2021)
LAMBDA_PSEUDO_MAX = 0.3
DANN_HIDDEN       = 64     # hidden width of the DANN/CDAN domain-classifier MLP

# v6/v7 lesson carried over: alignment losses only run on a capped number of steps per epoch,
# not once per source batch, or the small target set (364 images) gets replayed 8x/epoch and
# every technique above "succeeds" by memorising it.
DA_TARGET_PASSES  = 2

# ---------------- pseudo-labeling ----------------
PSEUDO_CONF            = 0.95
PSEUDO_TOPK_PER_CLASS  = 8      # class-balanced instead of a global threshold
PSEUDO_MIN_MONITOR_ACC = 0.85   # hard safety valve: below this, self-training switches itself off
PSEUDO_PLATEAU_EPOCHS  = 8      # only start once target-val has stopped improving
EMA_DECAY              = 0.999

# ---------------- knowledge distillation ----------------
LAMBDA_KD      = 1.0    # constant, not ramped, and not 3x the CE weight
LAMBDA_FEATKD  = 1.0
KD_TEMPERATURE = 4.0

SMOOTH_WINDOW = 5

CFG = dict(coral=LAMBDA_CORAL_MAX, mmd=LAMBDA_MMD_MAX, im=LAMBDA_IM_MAX, mcc=LAMBDA_MCC_MAX,
           dann=LAMBDA_DANN_MAX, cdan=LAMBDA_CDAN_MAX, dsan=LAMBDA_DSAN_MAX,
           pseudo=LAMBDA_PSEUDO_MAX, kd=LAMBDA_KD, featkd=LAMBDA_FEATKD, T=KD_TEMPERATURE,
           da_passes=DA_TARGET_PASSES, pseudo_conf=PSEUDO_CONF,
           pseudo_topk=PSEUDO_TOPK_PER_CLASS, pseudo_min_acc=PSEUDO_MIN_MONITOR_ACC)

print(f"\nDA_METHOD (this notebook): {DA_METHOD.upper()}")
print(f"IMG={IMG_SIZE} channels={STUDENT_CHANNELS} BS={BATCH_SIZE}/{TARGET_BATCH_SIZE}")
print(f"Active DA lambda: {DA_METHOD}={CFG[DA_METHOD]}  (all other DA lambdas unused this run)")
print(f"KD: lambda={LAMBDA_KD} (constant) T={KD_TEMPERATURE} feat={LAMBDA_FEATKD}")
print(f"Pseudo: teacher={TEACHER_USE_PSEUDO} student={STUDENT_USE_PSEUDO} "
      f"conf>={PSEUDO_CONF} topk={PSEUDO_TOPK_PER_CLASS}/class auto-off<{PSEUDO_MIN_MONITOR_ACC}")


Device: cuda | PyTorch: 2.10.0+cu128
GPU: Tesla T4

DA_METHOD (this notebook): CDAN
IMG=128 channels=(32, 48, 64, 96, 96) BS=48/32
Active DA lambda: cdan=1.0  (all other DA lambdas unused this run)
KD: lambda=1.0 (constant) T=4.0 feat=1.0
Pseudo: teacher=False student=True conf>=0.95 topk=8/class auto-off<0.85


---
# Part 1: Data — indexing, splits, domain-gap augmentation

Three things happen here, in order:

1. **Indexing** — walk both dataset roots, match folder-name aliases per domain, build
   `img_paths` / `labels`. Then one stratified 70/15/15 split per domain, computed once and
   reused by every model so teacher and student are always evaluated on identical images.
2. **Background randomization (FIX #6)** — PlantVillage is a centred leaf on a uniform grey
   backdrop; PlantDoc is a leaf somewhere in a cluttered field. A model trained on the former
   learns "uniform background => confident prediction" and, as published Grad-CAM analyses of
   this exact shift show, moves its attention onto the background at test time. So: recover the
   leaf mask (from PlantVillage's own `segmented/` split where available, otherwise a
   border-colour heuristic) and composite the leaf onto random crops of **PlantDoc train**
   images. Target val/test images are never used as backgrounds.
3. **`DualResDataset` (FIX #4)** — the reason v6's distillation failed. v6 augmented at 96px and
   then bilinearly upsampled to 224px for the teacher, so the teacher — fine-tuned on real 224px
   photos — was scored on blurry stretched inputs and returned near-useless soft targets, which
   were then weighted 3x the classification loss. Here a single geometric+photometric
   augmentation is applied once at 256px, then rendered twice: 128px/[-1,1] for the student and
   224px/ImageNet-norm for the teacher. Same crop, same jitter, same pixels, correct scale for both.

Every sample is a 5-tuple `(x_student, x_teacher, x_weak, label, domain)`. `x_weak` is a
resize+flip-only view used solely to *generate* pseudo-labels, while the model trains on the
strongly augmented `x_student` — the weak/strong split that makes FixMatch-style self-training
stable. Unused views are returned as a 1-element placeholder so collation stays uniform.

In [ ]:
# ============================================================
# PART 1A: DATASET INDEX
# ============================================================
SOURCE_ALL    = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"
SEGMENTED_ALL = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/segmented"
TARGET_ALL    = "/kaggle/input/datasets/nirmalsankalana/plantdoc-dataset"

SOURCE_CLASS_DIRS = {
    "Tomato_Septoria_Leaf_Spot": ["Tomato___Septoria_leaf_spot"],
    "Potato_Early_Blight":       ["Potato___Early_blight"],
    "Corn_Leaf_Blight":          ["Corn_(maize)___Northern_Leaf_Blight"],
}
TARGET_CLASS_DIRS = {
    "Tomato_Septoria_Leaf_Spot": ["Tomato Septoria leaf spot"],
    "Potato_Early_Blight":       ["Potato leaf early blight"],
    "Corn_Leaf_Blight":          ["Corn leaf blight"],
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


class LeafIndex:
    """Path/label index only -- no transforms. Walks root_dir recursively and pools
    every image under any folder whose normalized name matches an alias for a class."""

    def __init__(self, root_dir, classes, class_dir_map, domain_label=0, name=""):
        self.img_paths, self.labels = [], []
        self.classes, self.domain_label = classes, domain_label

        def _norm(s):
            return s.strip().lower().replace(" ", "").replace("_", "")

        for idx, cls in enumerate(classes):
            alias = {_norm(a) for a in class_dir_map.get(cls, [cls])}
            matched = [dp for dp, _, _ in os.walk(root_dir) if _norm(os.path.basename(dp)) in alias]
            for d in matched:
                for f in sorted(os.listdir(d)):
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                        self.img_paths.append(os.path.join(d, f)); self.labels.append(idx)
            if not matched:
                print(f"    WARNING: no folder matched '{cls}' under {root_dir}")

        print(f"  [{name}] {len(self.img_paths)} images")
        for i, c in enumerate(classes):
            print(f"    {c}: {self.labels.count(i)}")

    def subset(self, indices):
        s = copy.copy(self)
        s.img_paths = [self.img_paths[i] for i in indices]
        s.labels    = [self.labels[i] for i in indices]
        return s


def stratified_split_indices(labels, val_frac=0.15, test_frac=0.15, seed=SEED):
    idx_all = list(range(len(labels)))
    idx_tv, idx_test = train_test_split(idx_all, test_size=test_frac, stratify=labels,
                                        random_state=seed)
    lab_tv = [labels[i] for i in idx_tv]
    idx_tr, idx_val = train_test_split(idx_tv, test_size=val_frac / (1 - test_frac),
                                       stratify=lab_tv, random_state=seed)
    return idx_tr, idx_val, idx_test


for nm, p in [('PlantVillage color', SOURCE_ALL), ('PlantVillage segmented', SEGMENTED_ALL),
              ('PlantDoc', TARGET_ALL)]:
    print(f"  {nm:24s}: {'OK ' if os.path.isdir(p) else '!! NOT FOUND !!'} {p}")

print("\n--- Source (PlantVillage) ---")
src_index = LeafIndex(SOURCE_ALL, CLASSES, SOURCE_CLASS_DIRS, 0, "Source")
print("\n--- Target (PlantDoc) ---")
tgt_index = LeafIndex(TARGET_ALL, CLASSES, TARGET_CLASS_DIRS, 1, "Target")

src_tr_i, src_va_i, src_te_i = stratified_split_indices(src_index.labels)
tgt_tr_i, tgt_va_i, tgt_te_i = stratified_split_indices(tgt_index.labels)

src_train_idx = src_index.subset(src_tr_i)
src_val_idx   = src_index.subset(src_va_i)
src_test_idx  = src_index.subset(src_te_i)
tgt_train_idx = tgt_index.subset(tgt_tr_i)
tgt_val_idx   = tgt_index.subset(tgt_va_i)
tgt_test_idx  = tgt_index.subset(tgt_te_i)

print(f"\nSource  train/val/test: {len(src_train_idx.labels)}/"
      f"{len(src_val_idx.labels)}/{len(src_test_idx.labels)}")
print(f"Target  train/val/test: {len(tgt_train_idx.labels)}/"
      f"{len(tgt_val_idx.labels)}/{len(tgt_test_idx.labels)}")
print(f"\nNOTE: target test = {len(tgt_test_idx.labels)} images -> 1 image = "
      f"{100/len(tgt_test_idx.labels):.2f} accuracy points. Never quote a single point estimate.")

  PlantVillage color      : OK  /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color
  PlantVillage segmented  : OK  /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/segmented
  PlantDoc                : OK  /kaggle/input/datasets/nirmalsankalana/plantdoc-dataset

--- Source (PlantVillage) ---
  [Source] 3756 images
    Tomato_Septoria_Leaf_Spot: 1771
    Potato_Early_Blight: 1000
    Corn_Leaf_Blight: 985

--- Target (PlantDoc) ---
  [Target] 522 images
    Tomato_Septoria_Leaf_Spot: 157
    Potato_Early_Blight: 171
    Corn_Leaf_Blight: 194

Source  train/val/test: 2628/564/564
Target  train/val/test: 364/79/79

NOTE: target test = 79 images -> 1 image = 1.27 accuracy points. Never quote a single point estimate.


In [ ]:
# ============================================================
# PART 1B: BACKGROUND RANDOMIZATION  (FIX #6)
# ============================================================
# PlantVillage: centred leaf, uniform grey backdrop. PlantDoc: leaf in a cluttered field.
# A source-only model latches onto the backdrop as a shortcut. Cutting the leaf out and
# pasting it onto real field backgrounds removes that shortcut at the data level, which is
# more effective than any loss-side alignment term.
#
# Mask source, in order of preference:
#   1. PlantVillage's own segmented/ split (exact, free) -- matched by filename stem
#   2. Border-colour heuristic + morphological cleanup (works because the backdrop is uniform)
# Backgrounds: random crops of PlantDoc TRAIN images only (never val/test -> no leakage).

from scipy import ndimage as ndi


def build_segmented_lookup(seg_root, class_dir_map, classes):
    """stem -> segmented path. PlantVillage segmented filenames carry a '_final_masked'
    suffix, so match on the leading UUID-ish stem rather than the full filename."""
    lut = {}
    if not (seg_root and os.path.isdir(seg_root)):
        return lut

    def _norm(s):
        return s.strip().lower().replace(" ", "").replace("_", "")

    alias = set()
    for c in classes:
        alias |= {_norm(a) for a in class_dir_map.get(c, [c])}
    for dp, _, files in os.walk(seg_root):
        if _norm(os.path.basename(dp)) not in alias:
            continue
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                stem = os.path.splitext(f)[0].replace('_final_masked', '')
                lut[stem] = os.path.join(dp, f)
    return lut


SEG_LUT = build_segmented_lookup(SEGMENTED_ALL, SOURCE_CLASS_DIRS, CLASSES) if USE_BG_RANDOMIZATION else {}
print(f"Segmented-mask lookup: {len(SEG_LUT)} entries "
      f"({'exact masks available' if SEG_LUT else 'falling back to the border heuristic'})")


def estimate_leaf_mask(img, seg_path=None):
    """Boolean HxW mask, True = leaf. Returns None when the result looks untrustworthy,
    in which case the caller simply skips background randomization for that image."""
    if seg_path is not None and os.path.exists(seg_path):
        try:
            seg = np.asarray(Image.open(seg_path).convert("RGB").resize(img.size), np.float32)
            m = seg.sum(2) > 24          # segmented split paints the background pure black
            if 0.05 < m.mean() < 0.97:
                return ndi.binary_closing(m, np.ones((5, 5)))
        except Exception:
            pass

    a = np.asarray(img.convert("RGB"), np.float32)
    b = np.concatenate([a[:6].reshape(-1, 3), a[-6:].reshape(-1, 3),
                        a[:, :6].reshape(-1, 3), a[:, -6:].reshape(-1, 3)])
    med = np.median(b, 0)
    d = np.linalg.norm(a - med, axis=2)
    m = d > max(35.0, np.percentile(d, 40))
    m = ndi.binary_opening(m, np.ones((3, 3)))
    m = ndi.binary_closing(m, np.ones((9, 9)))
    m = ndi.binary_fill_holes(m)

    lab, n = ndi.label(m)                # keep only the largest blob (the leaf)
    if n > 1:
        sizes = ndi.sum(m, lab, range(1, n + 1))
        m = lab == (int(np.argmax(sizes)) + 1)
    if not (0.05 < m.mean() < 0.95):
        return None
    return m


def build_background_bank(index_obj, n=180, size=320):
    """Field backgrounds taken from PlantDoc TRAIN images only."""
    rng = np.random.RandomState(SEED)
    paths = list(index_obj.img_paths)
    rng.shuffle(paths)
    bank = []
    for p in paths[:n]:
        try:
            bank.append(Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR))
        except Exception:
            continue
    return bank


BG_BANK = build_background_bank(tgt_train_idx) if USE_BG_RANDOMIZATION else []
print(f"Background bank: {len(BG_BANK)} field images (PlantDoc TRAIN split only)")


MASK_SIZE = 192


def precompute_masks(index_obj):
    """Masks are computed ONCE here and stored bit-packed (~6 KB each), instead of being
    recomputed inside the dataloader on every __getitem__. Without this the scipy
    morphology becomes the training bottleneck."""
    if not USE_BG_RANDOMIZATION:
        return {}
    cache, t0 = {}, time.time()
    for n, p in enumerate(index_obj.img_paths):
        try:
            im = Image.open(p).convert("RGB").resize((MASK_SIZE, MASK_SIZE), Image.BILINEAR)
            stem = os.path.splitext(os.path.basename(p))[0]
            m = estimate_leaf_mask(im, SEG_LUT.get(stem))
            if m is not None:
                cache[p] = np.packbits(m)
        except Exception:
            pass
        if (n + 1) % 500 == 0:
            print(f"    masks {n+1}/{len(index_obj.img_paths)} ({time.time()-t0:.0f}s)")
    print(f"  usable leaf masks: {len(cache)}/{len(index_obj.img_paths)} "
          f"in {time.time()-t0:.0f}s")
    return cache


MASK_CACHE = precompute_masks(src_train_idx) if USE_BG_RANDOMIZATION else {}


def randomize_background(img, path, rng):
    """Composite the leaf onto a random field background with a soft alpha edge."""
    if not BG_BANK or path not in MASK_CACHE:
        return img
    m = np.unpackbits(MASK_CACHE[path])[:MASK_SIZE * MASK_SIZE].reshape(MASK_SIZE, MASK_SIZE)

    bg = BG_BANK[rng.randint(len(BG_BANK))]
    W, H = img.size
    bw, bh = bg.size
    cw = rng.randint(int(bw * 0.4), bw + 1)
    x0 = rng.randint(0, bw - cw + 1); y0 = rng.randint(0, bh - cw + 1)
    bg = bg.crop((x0, y0, x0 + cw, y0 + cw)).resize((W, H), Image.BILINEAR)

    alpha = (Image.fromarray((m * 255).astype(np.uint8))
             .resize((W, H), Image.BILINEAR).filter(ImageFilter.GaussianBlur(1.5)))
    out = Image.composite(img, bg, alpha)
    if rng.rand() < 0.3:                 # sometimes blur the background only
        out = Image.composite(out, bg.filter(ImageFilter.GaussianBlur(2.0)), alpha)
    return out

Segmented-mask lookup: 3756 entries (exact masks available)
Background bank: 180 field images (PlantDoc TRAIN split only)
    masks 500/2628 (13s)
    masks 1000/2628 (26s)
    masks 1500/2628 (39s)
    masks 2000/2628 (53s)
    masks 2500/2628 (66s)
  usable leaf masks: 2628/2628 in 69s


In [ ]:
# ============================================================
# PART 1C: DualResDataset  (FIX #4 + FIX #6)
# ============================================================
# One augmentation, rendered at both resolutions. This is the cell that repairs distillation:
# v6 augmented at 96px and bilinearly upsampled to 224px for the teacher, so the teacher saw
# blurry stretched inputs it was never fine-tuned on and emitted soft targets that were then
# weighted 3x the CE loss. Here both views come from the same 256px augmented image.
#
# Returns (x_student, x_teacher, x_weak_student, x_weak_teacher, label, domain).
# Two weak views are needed because pseudo-labels must be generated at whatever resolution the
# model being trained consumes -- 128px for the student, 224px for the teacher.
# Unrequested views are 1-element placeholders so the default collate still stacks cleanly.

_PLACEHOLDER = torch.zeros(1)


class DualResDataset(Dataset):
    def __init__(self, index_obj, train=True, emit_teacher=False, emit_weak=False,
                 bg_random_p=0.0, img_size=IMG_SIZE, teacher_size=TEACHER_IMG_SIZE):
        self.idx = index_obj
        self.train = train
        self.emit_teacher = emit_teacher
        self.emit_weak = emit_weak
        self.bg_p = bg_random_p if index_obj.domain_label == 0 else 0.0
        self.img_size, self.teacher_size = img_size, teacher_size

        # -- strong, domain-gap-aware augmentation (FIX #6) --
        # RandomResizedCrop(0.25-1.0) is the important one: it forces the model onto lesion
        # texture instead of whole-leaf silhouette, which does not survive the domain shift.
        self.geo_train = transforms.Compose([
            transforms.RandomResizedCrop(256, scale=(0.25, 1.0), ratio=(0.75, 1.33)),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.2),
            transforms.RandomApply([transforms.RandomRotation(25)], p=0.5),
            transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            transforms.RandomGrayscale(p=0.1),
            transforms.RandomApply([transforms.GaussianBlur(5, (0.1, 2.0))], p=0.3),
        ])
        self.geo_eval = transforms.Resize((256, 256))
        self.geo_weak = transforms.Compose([transforms.Resize((256, 256)),
                                            transforms.RandomHorizontalFlip(0.5)])

        self.to_student = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3)])
        self.to_teacher = transforms.Compose([
            transforms.Resize((teacher_size, teacher_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
        self.erase = transforms.RandomErasing(p=0.25, scale=(0.02, 0.15), value='random')

    def __len__(self):
        return len(self.idx.img_paths)

    def __getitem__(self, i):
        path = self.idx.img_paths[i]
        img = Image.open(path).convert("RGB")
        y, d = self.idx.labels[i], self.idx.domain_label

        if self.train and self.bg_p > 0:
            rng = np.random.RandomState((i * 7919 + int(time.time() * 1e3)) % (2 ** 31))
            if rng.rand() < self.bg_p:
                img = randomize_background(img.resize((256, 256), Image.BILINEAR), path, rng)

        base = self.geo_train(img) if self.train else self.geo_eval(img)

        xs = self.to_student(base)
        if self.train:
            xs = self.erase(xs)
        xt = self.to_teacher(base) if self.emit_teacher else _PLACEHOLDER
        if self.emit_weak:
            wbase = self.geo_weak(img)
            xw_s = self.to_student(wbase)
            xw_t = self.to_teacher(wbase) if self.emit_teacher else _PLACEHOLDER
        else:
            xw_s = xw_t = _PLACEHOLDER
        return xs, xt, xw_s, xw_t, y, d

In [ ]:
# ============================================================
# PART 1D: DATASETS AND LOADERS
# ============================================================
NW = 4          # lower to 2 if the Kaggle worker runs out of RAM

def mk_loader(ds, bs, shuffle, drop_last=False):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=NW,
                      pin_memory=True, drop_last=drop_last, persistent_workers=(NW > 0))

BG_P = 0.5 if USE_BG_RANDOMIZATION else 0.0

# -- train --
src_train_ds = DualResDataset(src_train_idx, True,  emit_teacher=True, bg_random_p=BG_P)
tgt_train_ds = DualResDataset(tgt_train_idx, True,  emit_teacher=True, emit_weak=True)

# -- eval (no augmentation anywhere) --
src_val_ds   = DualResDataset(src_val_idx,   False, emit_teacher=True)
src_test_ds  = DualResDataset(src_test_idx,  False, emit_teacher=True)
tgt_val_ds   = DualResDataset(tgt_val_idx,   False, emit_teacher=True)
tgt_test_ds  = DualResDataset(tgt_test_idx,  False, emit_teacher=True)

# FIX #12: v6 measured "source train accuracy" through the augmented training loader, which
# understated it and therefore corrupted the reported generalization gap. Clean loader:
src_train_eval_ds = DualResDataset(src_train_idx, False, emit_teacher=True)

src_train_loader = mk_loader(src_train_ds, BATCH_SIZE, True,  drop_last=True)
# FIX #7: drop_last=False -- v6 silently discarded 44 of 364 target images every epoch.
tgt_train_loader = mk_loader(tgt_train_ds, TARGET_BATCH_SIZE, True, drop_last=False)

src_val_loader        = mk_loader(src_val_ds,   BATCH_SIZE, False)
src_test_loader       = mk_loader(src_test_ds,  BATCH_SIZE, False)
tgt_val_loader        = mk_loader(tgt_val_ds,   BATCH_SIZE, False)
tgt_test_loader       = mk_loader(tgt_test_ds,  BATCH_SIZE, False)
src_train_eval_loader = mk_loader(src_train_eval_ds, BATCH_SIZE, False)

print(f"source batches/epoch : {len(src_train_loader)}")
print(f"target batches/epoch : {len(tgt_train_loader)}")
print(f"DA steps/epoch       : {min(len(src_train_loader), DA_TARGET_PASSES*len(tgt_train_loader))}"
      f"  -> each target image seen ~{DA_TARGET_PASSES}x/epoch")
print(f"  (v6 cycled the target loader {len(src_train_loader)/max(len(tgt_train_loader),1):.1f}x "
      f"per epoch -- that over-weighting is what drove CORAL to 1e-4 with no accuracy gain)")

xb = next(iter(src_train_loader))
print(f"\nbatch shapes: student={tuple(xb[0].shape)} teacher={tuple(xb[1].shape)}")
print(f"student range: [{xb[0].min():.2f}, {xb[0].max():.2f}]  (expect ~[-1, 1])")
xt_b = next(iter(tgt_train_loader))
print(f"target weak views: student={tuple(xt_b[2].shape)} teacher={tuple(xt_b[3].shape)}")

source batches/epoch : 54
target batches/epoch : 12
DA steps/epoch       : 24  -> each target image seen ~2x/epoch
  (v6 cycled the target loader 4.5x per epoch -- that over-weighting is what drove CORAL to 1e-4 with no accuracy gain)

batch shapes: student=(48, 3, 128, 128) teacher=(48, 3, 224, 224)
student range: [-3.86, 4.09]  (expect ~[-1, 1])
target weak views: student=(32, 3, 128, 128) teacher=(32, 3, 224, 224)


In [ ]:
# ============================================================
# PART 1E: VISUAL CHECK -- is background randomization actually working?
# ============================================================
# Worth eyeballing before a long run. If the composites look wrong (leaf cut in half, mask
# inverted), set USE_BG_RANDOMIZATION=False rather than training on corrupted images.

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
fig.suptitle("Row 1-2: source, augmented (background randomized) | Row 3: target (PlantDoc)",
             fontsize=14, fontweight='bold')

vis_ds = DualResDataset(src_train_idx, True, bg_random_p=1.0)
for k in range(6):
    for r in range(2):
        x = vis_ds[k * 3 + r][0]
        axes[r][k].imshow(np.clip(x.permute(1, 2, 0).numpy() * 0.5 + 0.5, 0, 1))
        axes[r][k].axis('off')

vis_t = DualResDataset(tgt_train_idx, False)
for k in range(6):
    x = vis_t[k * 5][0]
    axes[2][k].imshow(np.clip(x.permute(1, 2, 0).numpy() * 0.5 + 0.5, 0, 1))
    axes[2][k].axis('off')

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/augmentation_check.png", dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, idxo, t, c in [(axes[0], src_train_idx, 'Source train', '#2196F3'),
                       (axes[1], tgt_train_idx, 'Target train', '#FF5722')]:
    cnt = [idxo.labels.count(i) for i in range(NUM_CLASSES)]
    ax.bar(range(NUM_CLASSES), cnt, color=c, alpha=.85)
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_xticklabels([c2[:14] for c2 in CLASSES], rotation=15, fontsize=9)
    ax.set_title(t, fontweight='bold')
    for i, v in enumerate(cnt):
        ax.text(i, v, str(v), ha='center', va='bottom')
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

shares = np.array([src_index.labels.count(i) for i in range(NUM_CLASSES)], float)
shares /= shares.sum()
print("Source class shares:", {CLASSES[i]: round(float(shares[i]), 4) for i in range(NUM_CLASSES)})
print(f"\nCOLLAPSE REFERENCE -- if source val accuracy parks at one of these, a class has been")
print(f"erased from the feature space (this is exactly what v6's TA did at 0.730):")
for i in range(NUM_CLASSES):
    print(f"  all of {CLASSES[i][:22]:<22s} absorbed elsewhere -> {1 - shares[i]:.4f}")

Source class shares: {'Tomato_Septoria_Leaf_Spot': 0.4715, 'Potato_Early_Blight': 0.2662, 'Corn_Leaf_Blight': 0.2622}

COLLAPSE REFERENCE -- if source val accuracy parks at one of these, a class has been
erased from the feature space (this is exactly what v6's TA did at 0.730):
  all of Tomato_Septoria_Leaf_S absorbed elsewhere -> 0.5285
  all of Potato_Early_Blight    absorbed elsewhere -> 0.7338
  all of Corn_Leaf_Blight       absorbed elsewhere -> 0.7378


---
# Part 2: Architecture

Same three-stage idea as v6 — CNN → Squeeze-Excite → Micro-ViT with linear attention — with
three changes:

- **Wider and higher-resolution (FIX #8).** `(32,48,64,96,96)` at 128px instead of
  `(16,24,32,48,48)` at 96px -- 167K params, ~163 KB INT8. v6 shipped 42 KB against a stated 500 KB budget; the two
  classes it confused (Septoria's small scattered spots vs Early Blight's concentric rings) are
  separated by exactly the fine texture that 96px destroys. The extra stride-2 stage keeps the
  token grid at 8×8 = 64, so linear attention stays cheap.
- **Learned positional embedding.** v6's Micro-ViT had none, so its attention was
  permutation-invariant over patches — lesion *distribution* across the leaf was invisible to it.
- **`kd_proj` lives outside the model.** The feature-distillation projection is a separate
  module, so the deployed graph is untouched and nothing needs stripping before export.

`extract_features()` is the DA hook: CORAL/MMD/MCC all operate on its output.

In [ ]:
# ============================================================
# PART 2: MicroBrainNet-DA v7
# ============================================================

class InvertedResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, expand_ratio=2):
        super().__init__()
        self.use_residual = (stride == 1 and in_ch == out_ch)
        hid = in_ch * expand_ratio
        layers = []
        if expand_ratio != 1:
            layers += [nn.Conv2d(in_ch, hid, 1, bias=False), nn.BatchNorm2d(hid),
                       nn.ReLU6(inplace=True)]
        layers += [nn.Conv2d(hid, hid, 3, stride, 1, groups=hid, bias=False),
                   nn.BatchNorm2d(hid), nn.ReLU6(inplace=True),
                   nn.Conv2d(hid, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch)]
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.conv(x) if self.use_residual else self.conv(x)


class SqueezeExciteBlock(nn.Module):
    def __init__(self, ch, reduction=4):
        super().__init__()
        mid = max(ch // reduction, 8)
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excite = nn.Sequential(nn.Linear(ch, mid), nn.ReLU(inplace=True),
                                    nn.Linear(mid, ch), nn.Sigmoid())

    def forward(self, x):
        b, c, _, _ = x.shape
        return x * self.excite(self.squeeze(x).view(b, c)).view(b, c, 1, 1)


class MicroViTBlock(nn.Module):
    """Linear attention, O(n). ReLU6 in the FFN (not GELU) so every op has a TFLite Micro kernel."""

    def __init__(self, dim, num_tokens, ffn_ratio=2):
        super().__init__()
        self.q_proj = nn.Linear(dim, dim); self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim); self.out_proj = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim); self.norm2 = nn.LayerNorm(dim)
        # learned positional embedding -- v6 had none, so patch layout was invisible
        self.pos = nn.Parameter(torch.zeros(1, num_tokens, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        h = dim * ffn_ratio
        self.ffn = nn.Sequential(nn.Linear(dim, h), nn.ReLU6(inplace=True), nn.Dropout(0.1),
                                 nn.Linear(h, dim), nn.Dropout(0.1))

    def linear_attention(self, q, k, v):
        q = F.elu(q) + 1
        k = F.elu(k) + 1
        kv = torch.bmm(k.transpose(1, 2), v)
        num = torch.bmm(q, kv)
        den = torch.bmm(q, k.sum(1, keepdim=True).transpose(1, 2)).clamp(min=1e-4)
        return num / den

    def forward(self, x):
        x = x + self.pos
        xn = self.norm1(x)
        x = x + self.out_proj(self.linear_attention(self.q_proj(xn), self.k_proj(xn),
                                                    self.v_proj(xn)))
        return x + self.ffn(self.norm2(x))


class MicroBrainNetDA(nn.Module):
    """128x128x3 RGB -> 3 logits. Downsampling: stem s2, b1 s2, b2 s2, b3 s2, b4 s1
    => 8x8 = 64 tokens for the ViT stage."""

    def __init__(self, num_classes=NUM_CLASSES, channels=STUDENT_CHANNELS, img_size=IMG_SIZE,
                 use_se=True, use_vit=True):
        super().__init__()
        c0, c1, c2, c3, c4 = channels
        self.use_se, self.use_vit = use_se, use_vit
        self.stem = nn.Sequential(nn.Conv2d(3, c0, 3, 2, 1, bias=False),
                                  nn.BatchNorm2d(c0), nn.ReLU6(inplace=True))
        self.block1 = InvertedResidualBlock(c0, c1, 2, 2)
        self.block2 = InvertedResidualBlock(c1, c2, 2, 2)
        self.block3 = InvertedResidualBlock(c2, c3, 2, 2)
        self.block4 = InvertedResidualBlock(c3, c4, 1, 2)
        self.se = SqueezeExciteBlock(c4) if use_se else nn.Identity()
        self.grid = img_size // 16
        self.vit = MicroViTBlock(c4, self.grid * self.grid) if use_vit else nn.Identity()
        self.classifier = nn.Sequential(nn.LayerNorm(c4), nn.Dropout(0.2),
                                        nn.Linear(c4, num_classes))
        self.feature_dim = c4
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def extract_features(self, x):
        x = self.stem(x)
        x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.block4(x)
        x = self.se(x)
        if self.use_vit:
            x = x.flatten(2).transpose(1, 2)
            x = self.vit(x)
            return x.mean(1)
        return F.adaptive_avg_pool2d(x, 1).flatten(1)

    def forward(self, x):
        return self.classifier(self.extract_features(x))


class TeacherNet(nn.Module):
    """ImageNet-pretrained backbone behind the same extract_features()/classifier interface,
    so it drops straight into train_da_epoch(). Training-time only -- never quantized."""

    def __init__(self, num_classes=NUM_CLASSES, backbone_name=TEACHER_BACKBONE):
        super().__init__()
        if backbone_name == 'mobilenet_v3_large':
            bb = tvm.mobilenet_v3_large(weights=tvm.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
        elif backbone_name == 'mobilenet_v3_small':
            bb = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        elif backbone_name == 'efficientnet_b0':
            bb = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1)
        else:
            raise ValueError(backbone_name)
        self.features = bb.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.feature_dim = bb.classifier[0].in_features
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(self.feature_dim, num_classes))

    def extract_features(self, x):
        return self.pool(self.features(x)).flatten(1)

    def forward(self, x):
        return self.classifier(self.extract_features(x))


model = MicroBrainNetDA().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Student params : {total_params:,}")
print(f"  FP32 : {total_params*4/1024:.1f} KB")
print(f"  INT8 : {total_params/1024:.1f} KB   (budget 500 KB -> "
      f"{total_params/1024/500*100:.0f}% used; v6 used 8%)")
print(f"  ViT token grid: {model.grid}x{model.grid} = {model.grid**2} tokens")
with torch.no_grad():
    print("forward:", tuple(model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)).shape))

Student params : 167,131
  FP32 : 652.9 KB
  INT8 : 163.2 KB   (budget 500 KB -> 33% used; v6 used 8%)
  ViT token grid: 8x8 = 64 tokens
forward: (2, 3)


---
# Part 3: Losses

This cell defines **all seven** candidate alignment losses (CORAL, MMD, IM, MCC, DANN, CDAN,
DSAN) so its text is identical in every comparison notebook. `DA_METHOD` (Part 0) is the only
thing that decides which one actually gets a nonzero weight in `train_da_epoch` (Part 4) — see
the `use_coral / use_mmd / use_im / use_mcc / use_dann / use_cdan / use_dsan` flags there, which
are all derived from the single `DA_METHOD` string.

**This notebook's active technique: CDAN — Conditional Domain Adversarial Network, CDAN+E (Long et al., 2018).**
Like DANN, but the domain classifier is conditioned on the multilinear map of feature and predicted class, with entropy-based sample weighting, so alignment is informed by class structure instead of being class-agnostic.

### Statistic-matching family (CORAL, MMD, DSAN)
`coral_loss` matches second-order feature covariance; `mmd_loss` matches a multi-kernel mean
embedding of the marginal feature distribution. Both are class-agnostic: a feature map that
throws away class structure can still satisfy them (the classic "degenerate alignment" failure).
`dsan_local_mmd` fixes exactly this by matching **class-conditional** (subdomain) distributions
instead — source samples weighted by true label, target samples weighted by predicted class
probability.

### Target-regularization family (IM, MCC)
`im_loss` (Information Maximization) keeps the confidence term of entropy minimization but
subtracts the entropy of the batch-mean prediction, so collapsing every target sample to one
class is the *worst* outcome instead of the best. `mcc_loss` (Minimum Class Confusion) penalises
off-diagonal mass in the target's own class-confusion matrix, so it likewise cannot be satisfied
by a collapse.

### Adversarial family (DANN, CDAN)
`GradReverse` + `DomainClassifier` implement the adversarial two-player game: a small MLP tries
to tell source from target apart while the gradient reversal layer pushes the feature extractor
the opposite way. `dann_loss` runs this on raw features; `cdan_loss` conditions the discriminator
on the multilinear map feature ⊗ softmax(prediction) and applies CDAN+E entropy-based sample
weighting, so alignment is informed by class structure rather than class-agnostic.


In [ ]:
# ============================================================
# PART 3: LOSS FUNCTIONS
# ============================================================
# All seven candidate alignment techniques are defined here, in every notebook, so this cell's
# text is identical across the comparison set. Only DA_METHOD (Part 0) decides which one
# actually contributes to the loss during training -- see train_da_epoch in Part 4.

def standardize_feats(x, eps=1e-5):
    return (x - x.mean(0, keepdim=True)) / (x.std(0, keepdim=True) + eps)


def compute_covariance(x):
    n = x.size(0)
    ones = torch.ones(1, n, device=x.device)
    mean = ones @ x / n
    return (x.t() @ x - mean.t() @ mean) / max(n - 1, 1)


def coral_loss(fs, ft):
    """Deep CORAL (Sun & Saenko, 2016): match second-order feature statistics (covariance)."""
    s, t = standardize_feats(fs), standardize_feats(ft)
    d = s.size(1)
    return (compute_covariance(s) - compute_covariance(t)).pow(2).sum() / (4.0 * d * d)


def gaussian_kernel_matrix(s, t, kernel_mul=2.0, kernel_num=5):
    total = torch.cat([s, t], 0)
    n = total.size(0)
    l2 = ((total.unsqueeze(0) - total.unsqueeze(1)) ** 2).sum(2)
    bw = torch.sum(l2.detach()) / max(n * n - n, 1)
    bw = bw / (kernel_mul ** (kernel_num // 2)) + 1e-8
    return sum(torch.exp(-l2 / (bw * kernel_mul ** i)) for i in range(kernel_num))


def mmd_loss(fs, ft):
    """Deep MMD (Gretton et al., 2012; Long et al., 2015): multi-kernel MMD on the marginal
    feature distributions."""
    s, t = standardize_feats(fs), standardize_feats(ft)
    ns = s.size(0)
    k = gaussian_kernel_matrix(s, t)
    return k[:ns, :ns].mean() + k[ns:, ns:].mean() - k[:ns, ns:].mean() - k[ns:, :ns].mean()


def im_loss(logits):
    """Information Maximization. Bare entropy minimization has one trivially optimal solution:
    send every target sample to the same class. Subtracting the entropy of the batch-mean
    prediction makes a collapsed solution the WORST outcome instead of the best.
    Returns ent_individual - ent_batch; both terms are minimized together."""
    p = F.softmax(logits, 1)
    ent_ind = -(p * torch.log(p + 1e-8)).sum(1).mean()
    pm = p.mean(0)
    ent_batch = -(pm * torch.log(pm + 1e-8)).sum()
    return ent_ind - ent_batch


def mcc_loss(logits, T=2.5):
    """Minimum Class Confusion (Jin et al., 2020). Class-aware, so unlike CORAL/MMD it cannot
    be satisfied by a feature collapse -- off-diagonal mass only falls if target predictions
    become mutually distinct."""
    p = F.softmax(logits / T, 1)
    w = (1 + (-(p * torch.log(p + 1e-8)).sum(1))).detach()
    w = (w / w.sum()) * p.size(0)
    C = (p * w.unsqueeze(1)).t() @ p
    C = C / (C.sum(1, keepdim=True) + 1e-8)
    return (C.sum() - C.trace()) / logits.size(1)


# ---------------- DANN / CDAN: adversarial alignment ----------------

class GradReverse(torch.autograd.Function):
    """Gradient Reversal Layer (Ganin & Lempitsky, 2015). Identity in the forward pass; flips
    and scales the gradient in the backward pass, so a single optimizer step pushes the domain
    classifier to DISCRIMINATE source vs target while pushing the feature extractor to CONFUSE
    it -- the two-player game that makes DANN/CDAN adversarial rather than a fixed-statistic
    alignment term like CORAL/MMD."""

    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lam, None


def grad_reverse(x, lam=1.0):
    return GradReverse.apply(x, lam)


class DomainClassifier(nn.Module):
    """Small MLP domain discriminator shared by DANN and CDAN. Training-time only -- like the
    teacher and kd_proj, it never enters the exported/deployed graph."""

    def __init__(self, in_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(hidden, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, 1))

    def forward(self, x):
        return self.net(x).squeeze(1)


def make_domain_clf(feature_dim, da_method, num_classes=NUM_CLASSES, hidden=DANN_HIDDEN):
    """Sized domain classifier for the active method, or None if the method needs none.
    CDAN conditions on feature (x) softmax(prediction); with feature_dim*num_classes this small
    (<=~3K here) an exact outer product is cheap, so no random projection is needed."""
    if da_method == 'dann':
        return DomainClassifier(feature_dim, hidden)
    if da_method == 'cdan':
        return DomainClassifier(feature_dim * num_classes, hidden)
    return None


def dann_loss(domain_clf, f_s, f_t, lam):
    """DANN (Ganin et al., 2016). Binary domain-adversarial loss on raw features."""
    f = torch.cat([f_s, f_t], 0)
    lbl = torch.cat([torch.zeros(f_s.size(0)), torch.ones(f_t.size(0))]).to(f.device)
    logits = domain_clf(grad_reverse(f, lam))
    return F.binary_cross_entropy_with_logits(logits, lbl)


def cdan_entropy_weight(logits):
    """CDAN+E per-sample weight (Long et al., 2018): down-weights target samples the classifier
    is uncertain about, so the adversarial signal leans on confident predictions."""
    p = F.softmax(logits, 1)
    ent = -(p * torch.log(p + 1e-8)).sum(1)
    w = 1.0 + torch.exp(-ent)
    return (w / w.mean()).detach()


def cdan_loss(domain_clf, f_s, f_t, logits_s, logits_t, lam):
    """CDAN (Long et al., 2018). The domain classifier sees the multilinear map
    feature (x) softmax(prediction) instead of raw features, so alignment is conditioned on
    class structure instead of being class-agnostic like DANN/CORAL/MMD."""
    f = torch.cat([f_s, f_t], 0)
    logits = torch.cat([logits_s, logits_t], 0)
    g = F.softmax(logits, 1)
    op = torch.bmm(g.unsqueeze(2), f.unsqueeze(1)).flatten(1)   # exact outer product
    lbl = torch.cat([torch.zeros(f_s.size(0)), torch.ones(f_t.size(0))]).to(f.device)
    dom_logits = domain_clf(grad_reverse(op, lam))
    ent_w = cdan_entropy_weight(logits)
    per_sample = F.binary_cross_entropy_with_logits(dom_logits, lbl, reduction='none')
    return (per_sample * ent_w).mean()


def dsan_local_mmd(fs, ft, ys, logits_t, n_classes=NUM_CLASSES):
    """DSAN / Local MMD (Zhu et al., 2021). CORAL/MMD match the MARGINAL feature distribution
    and are satisfied even if per-class structure is destroyed in the process -- the same
    degenerate-alignment failure mode IM/MCC above were built to catch. DSAN instead matches the
    CLASS-CONDITIONAL (subdomain) distributions directly: source samples are weighted by their
    true one-hot label, target samples by their softmax class probability, and a class-weighted
    MMD is summed over classes."""
    ys_oh = F.one_hot(ys, n_classes).float()
    pt = F.softmax(logits_t, 1)
    s, t = standardize_feats(fs), standardize_feats(ft)
    k = gaussian_kernel_matrix(s, t)
    ns = fs.size(0)
    kss, ktt, kst = k[:ns, :ns], k[ns:, ns:], k[:ns, ns:]
    loss = fs.new_zeros(())
    for c in range(n_classes):
        ws, wt = ys_oh[:, c], pt[:, c]
        ws = ws / (ws.sum() + 1e-8)
        wt = wt / (wt.sum() + 1e-8)
        loss = loss + (ws.unsqueeze(1) * ws.unsqueeze(0) * kss).sum() \
                     + (wt.unsqueeze(1) * wt.unsqueeze(0) * ktt).sum() \
                     - 2 * (ws.unsqueeze(1) * wt.unsqueeze(0) * kst).sum()
    return loss / n_classes


def kd_loss_fn(s_logits, t_logits, T=4.0):
    return F.kl_div(F.log_softmax(s_logits / T, 1), F.softmax(t_logits / T, 1),
                    reduction='batchmean') * (T * T)


def feat_kd_loss(s_proj, t_feat):
    """Cosine feature distillation. With a ~70x capacity gap, logit-only KD transfers little;
    matching normalized feature direction gives the student a much denser signal."""
    return F.mse_loss(F.normalize(s_proj, dim=1), F.normalize(t_feat.detach(), dim=1))


@torch.no_grad()
def balanced_pseudo(logits, k_per_class=PSEUDO_TOPK_PER_CLASS, min_conf=PSEUDO_CONF,
                    n_classes=NUM_CLASSES):
    """Class-balanced top-k selection. A global threshold lets the dominant class supply nearly
    every pseudo-label, which is how a poorly-behaved run reinforces its own errors."""
    probs = F.softmax(logits, 1)
    conf, lab = probs.max(1)
    keep = torch.zeros_like(conf, dtype=torch.bool)
    for c in range(n_classes):
        idx = ((lab == c) & (conf >= min_conf)).nonzero(as_tuple=True)[0]
        if idx.numel():
            top = conf[idx].topk(min(k_per_class, idx.numel())).indices
            keep[idx[top]] = True
    return keep, lab


def ramp(epoch, max_epoch, warm=0):
    """Sigmoid ramp for the alignment weight. KD is deliberately NOT ramped: ramping it up means
    it peaks exactly when it should be fading out."""
    if epoch <= warm:
        return 0.0
    p = (epoch - warm) / max(max_epoch - warm, 1)
    return float(2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0)


print("Losses ready: coral, mmd, im, mcc, dann, cdan, dsan, kd, feat_kd, balanced_pseudo")
print(f"Active this run: {DA_METHOD}")


Losses ready: coral, mmd, im, mcc, dann, cdan, dsan, kd, feat_kd, balanced_pseudo
Active this run: cdan


---
# Part 4: Training utilities

`train_da_epoch` is the single function used by the teacher (Part 5), the student (Part 5C),
multi-seed re-runs (Part 7) and the ablation (Part 9). It accepts `da_method` and, for the
adversarial techniques, a `domain_clf` module; in this notebook every call passes
`da_method=DA_METHOD` ("**cdan**"), so exactly one alignment loss is ever active.

### The BatchNorm fix

Source and target batches are concatenated into a **single forward pass** rather than run as two
separate `.train()`-mode passes, so BatchNorm sees one coherent mixed-domain distribution and
updates once per step instead of blending two very different running statistics. `adapt_bn()`
(AdaBN) then goes further: before final evaluation and before INT8 calibration it resets the
running statistics and recomputes them from target data only.

### The target over-sampling fix

`n_da_steps = min(len(src_loader), DA_TARGET_PASSES * len(tgt_loader))`. Beyond that many steps
the epoch continues on source CE (plus source-side KD) alone, so the 364-image target set cannot
be replayed many times per epoch and trivially "solved" by whichever alignment loss is active.

### Early stopping

Uses a 5-epoch rolling mean of target-val accuracy. On 79 images a single epoch's raw number
moves ~1.27 points per image, so unsmoothed selection just latches onto a lucky epoch.


In [ ]:
# ============================================================
# PART 4: TRAINING UTILITIES
# ============================================================

class EMA:
    """Exponential moving average of the student, used only to generate pseudo-labels."""

    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.model.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k].detach(), alpha=1 - self.decay)
            else:
                v.copy_(msd[k])


class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=1e-3):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best, self.early_stop, self.best_model = 0, None, False, None

    def __call__(self, score, model):
        if self.best is None or score > self.best + self.min_delta:
            self.best, self.counter = score, 0
            self.best_model = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


@torch.no_grad()
def evaluate(model, loader, criterion, device, which='s'):
    """which='s' -> 128px/[-1,1] student view; which='t' -> 224px/ImageNet teacher view."""
    model.eval()
    tot, preds, labs, probs = 0.0, [], [], []
    for xs, xt, xw_s, xw_t, y, d in loader:
        x = (xs if which == 's' else xt).to(device)
        y = y.to(device)
        logits = model(x)
        tot += criterion(logits, y).item() * x.size(0)
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.cpu().numpy())
        probs.extend(F.softmax(logits, 1).cpu().numpy())
    n = len(labs)
    return (tot / n, accuracy_score(labs, preds), f1_score(labs, preds, average='weighted'),
            np.array(preds), np.array(labs), np.array(probs))


@torch.no_grad()
def adapt_bn(model, loader, device, which='s', n_pass=3):
    """AdaBN: recompute BN running statistics from TARGET data only. momentum=None makes each
    BN accumulate a cumulative average over the passes. Must run before INT8 calibration,
    otherwise quantization bakes in the wrong statistics."""
    has_bn = False
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            m.reset_running_stats(); m.momentum = None; has_bn = True
    if not has_bn:
        return model
    model.train()
    for _ in range(n_pass):
        for xs, xt, xw_s, xw_t, y, d in loader:
            model((xs if which == 's' else xt).to(device))
    model.eval()
    return model


def train_da_epoch(model, src_loader, tgt_loader, criterion, optimizer, device,
                   epoch, max_epoch, cfg, which='s',
                   teacher=None, kd_proj=None, ema=None,
                   pseudo_enabled=False,
                   da_method='coral', domain_clf=None, da_active=True, use_kd=True):
    """One epoch of domain-adaptive (+ optionally distilled) training.

    da_method selects EXACTLY ONE alignment technique -- 'coral', 'mmd', 'im', 'mcc', 'dann',
    'cdan', or 'dsan' -- which is what makes the seven baseline notebooks directly comparable:
    this function's text is identical in every one of them, only the DA_METHOD config value
    (and therefore this argument) differs. da_active=False disables alignment entirely
    regardless of da_method (used by the ablation's 'w/o DA' row).

    Structural properties that hold for every technique:
      * source and target go through ONE concatenated forward -> clean BN statistics
      * alignment losses run on at most cfg['da_passes'] * len(tgt_loader) steps, so the small
        target set cannot be replayed many times per epoch and trivially "solved"
      * KD uses the paired native-resolution teacher view from DualResDataset
      * pseudo-labels come from the EMA model on the WEAK view, applied to the STRONG view
    """
    model.train()
    lam = ramp(epoch, max_epoch)
    use_coral = da_active and da_method == 'coral'
    use_mmd   = da_active and da_method == 'mmd'
    use_im    = da_active and da_method == 'im'
    use_mcc   = da_active and da_method == 'mcc'
    use_dann  = da_active and da_method == 'dann'
    use_cdan  = da_active and da_method == 'cdan'
    use_dsan  = da_active and da_method == 'dsan'

    w = dict(coral=cfg['coral'] * lam if use_coral else 0.0,
             mmd=cfg['mmd'] * lam if use_mmd else 0.0,
             im=cfg['im'] * lam if use_im else 0.0,
             mcc=cfg['mcc'] * lam if use_mcc else 0.0,
             dann=cfg['dann'] * lam if use_dann else 0.0,
             cdan=cfg['cdan'] * lam if use_cdan else 0.0,
             dsan=cfg['dsan'] * lam if use_dsan else 0.0,
             pseudo=cfg['pseudo'] * lam if pseudo_enabled else 0.0,
             kd=cfg['kd'] if (use_kd and teacher is not None) else 0.0,
             featkd=cfg['featkd'] if (use_kd and teacher is not None and kd_proj is not None) else 0.0)

    n_da = min(len(src_loader), max(1, cfg['da_passes'] * len(tgt_loader)))
    tgt_iter = iter(tgt_loader)

    acc_loss = {k: 0.0 for k in ['cls', 'coral', 'mmd', 'im', 'mcc', 'dann', 'cdan', 'dsan',
                                 'pseudo', 'kd', 'featkd']}
    n_seen, n_da_seen = 0, 0
    preds, labs = [], []
    ps_used, ps_correct = 0, 0

    for step, (xs_s, xs_t, _w1, _w2, ys, _d) in enumerate(src_loader):
        do_da = step < n_da
        x_src = (xs_s if which == 's' else xs_t).to(device, non_blocking=True)
        ys = ys.to(device, non_blocking=True)
        bs = x_src.size(0)

        if do_da:
            try:
                xt_s, xt_t, xt_ws, xt_wt, yt_true, _ = next(tgt_iter)
            except StopIteration:
                tgt_iter = iter(tgt_loader)
                xt_s, xt_t, xt_ws, xt_wt, yt_true, _ = next(tgt_iter)
            x_tgt = (xt_s if which == 's' else xt_t).to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # ---- single concatenated forward: clean BN statistics ----
        if do_da:
            feats = model.extract_features(torch.cat([x_src, x_tgt], 0))
            f_s, f_t = feats[:bs], feats[bs:]
            logits_s, logits_t = model.classifier(f_s), model.classifier(f_t)
        else:
            f_s = model.extract_features(x_src)
            f_t, logits_t = None, None
            logits_s = model.classifier(f_s)

        L_cls = criterion(logits_s, ys)
        loss = L_cls
        parts = {k: 0.0 for k in acc_loss}
        parts['cls'] = L_cls.item()

        if do_da:
            if w['coral'] > 0:
                l = coral_loss(f_s, f_t);  loss = loss + w['coral'] * l;  parts['coral'] = l.item()
            if w['mmd'] > 0:
                l = mmd_loss(f_s, f_t);    loss = loss + w['mmd'] * l;    parts['mmd'] = l.item()
            if w['im'] > 0:
                l = im_loss(logits_t);     loss = loss + w['im'] * l;     parts['im'] = l.item()
            if w['mcc'] > 0:
                l = mcc_loss(logits_t);    loss = loss + w['mcc'] * l;    parts['mcc'] = l.item()
            if w['dann'] > 0:
                l = dann_loss(domain_clf, f_s, f_t, lam)
                loss = loss + w['dann'] * l; parts['dann'] = l.item()
            if w['cdan'] > 0:
                l = cdan_loss(domain_clf, f_s, f_t, logits_s, logits_t, lam)
                loss = loss + w['cdan'] * l; parts['cdan'] = l.item()
            if w['dsan'] > 0:
                l = dsan_local_mmd(f_s, f_t, ys, logits_t)
                loss = loss + w['dsan'] * l; parts['dsan'] = l.item()

            if w['pseudo'] > 0 and ema is not None:
                with torch.no_grad():
                    # weak view at the resolution THIS model consumes -> pseudo-labels
                    xw = (xt_ws if which == 's' else xt_wt).to(device)
                    weak_logits = ema.model(xw)
                keep, plab = balanced_pseudo(weak_logits, cfg['pseudo_topk'], cfg['pseudo_conf'])
                if keep.sum() > 0:
                    l = F.cross_entropy(logits_t[keep], plab[keep])  # strong view -> trained on
                    loss = loss + w['pseudo'] * l
                    parts['pseudo'] = l.item()
                    ps_used += int(keep.sum())
                    ps_correct += int((plab[keep].cpu() == yt_true[keep.cpu()]).sum())

        # ---- distillation on the paired native-resolution teacher view ----
        if w['kd'] > 0:
            with torch.no_grad():
                tf_s = teacher.extract_features(xs_t.to(device, non_blocking=True))
                tl_s = teacher.classifier(tf_s)
                if do_da:
                    tf_t = teacher.extract_features(xt_t.to(device, non_blocking=True))
                    tl_t = teacher.classifier(tf_t)
            l = kd_loss_fn(logits_s, tl_s, cfg['T'])
            if do_da:
                l = 0.5 * (l + kd_loss_fn(logits_t, tl_t, cfg['T']))
            loss = loss + w['kd'] * l
            parts['kd'] = l.item()

            if w['featkd'] > 0:
                lf = feat_kd_loss(kd_proj(f_s), tf_s)
                if do_da:
                    lf = 0.5 * (lf + feat_kd_loss(kd_proj(f_t), tf_t))
                loss = loss + w['featkd'] * lf
                parts['featkd'] = lf.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if domain_clf is not None:
            torch.nn.utils.clip_grad_norm_(domain_clf.parameters(), 1.0)
        optimizer.step()
        if ema is not None:
            ema.update(model)

        for k in acc_loss:
            acc_loss[k] += parts[k] * bs
        n_seen += bs
        n_da_seen += bs if do_da else 0
        preds.extend(logits_s.argmax(1).detach().cpu().numpy())
        labs.extend(ys.cpu().numpy())

    out = {f'{k}_loss': v / max(n_seen, 1) for k, v in acc_loss.items()}
    out.update(train_acc=accuracy_score(labs, preds),
               train_f1=f1_score(labs, preds, average='weighted'),
               lam=lam, pseudo_used=ps_used,
               pseudo_acc=(ps_correct / ps_used) if ps_used else float('nan'),
               weights=w)
    return out


def make_scheduler(opt, epochs, warmup):
    def fn(e):
        return (e + 1) / max(warmup, 1) if e < warmup else \
               0.5 * (1 + math.cos(math.pi * (e - warmup) / max(epochs - warmup, 1)))
    return optim.lr_scheduler.LambdaLR(opt, fn)


cls_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
print("Utilities ready: EMA, EarlyStopping, evaluate, adapt_bn, train_da_epoch, make_scheduler")
print(f"train_da_epoch will use da_method='{DA_METHOD}' whenever called with the default")


Utilities ready: EMA, EarlyStopping, evaluate, adapt_bn, train_da_epoch, make_scheduler
train_da_epoch will use da_method='cdan' whenever called with the default


---
# Part 0B: Sanity gate — can the model fit SOURCE at all?

**Run this before anything else.** It trains the student and (optionally) a wide variant on
source only: no CORAL, no MMD, no IM, no MCC, no pseudo-labels, no distillation. It answers the
one question that determines whether any downstream tuning is meaningful.

| Result | Meaning | Next step |
|---|---|---|
| Src-Val **≥ 0.97** | Architecture and optimizer are fine. v6's 0.730 was caused by the training recipe, exactly as diagnosed. | Continue to Part 5. |
| Src-Val ≈ **0.73** (or any `1 - class_share` value printed in Part 1E) | A class is being erased even without DA. The problem is architectural or in the data. | Stop. Inspect the confusion matrix printed below before training anything else. |

The cell also prints train accuracy alongside val accuracy. v6 never logged the TA's train
accuracy, which is why it was impossible to tell "cannot fit the data" apart from "fits fine but
BN statistics are wrong at eval time" — the two require completely different fixes.

In [ ]:
# ============================================================
# PART 0B: SANITY GATE -- SOURCE-ONLY, NO DA, NO KD
# ============================================================
if RUN_SANITY_GATE:
    set_seed(SEED)
    print("=" * 72)
    print("  SANITY GATE: source-only training. Target >= 0.97 source-val accuracy.")
    print("=" * 72)

    sg_model = MicroBrainNetDA().to(DEVICE)
    sg_opt = optim.AdamW(sg_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sg_sched = make_scheduler(sg_opt, 30, 3)
    sg_crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    for ep in range(1, 31):
        sg_model.train()
        p_, l_ = [], []
        for xs, xt, xw_s, xw_t, y, d in src_train_loader:
            xs, y = xs.to(DEVICE), y.to(DEVICE)
            sg_opt.zero_grad(set_to_none=True)
            out = sg_model(xs)
            loss = sg_crit(out, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(sg_model.parameters(), 1.0)
            sg_opt.step()
            p_.extend(out.argmax(1).detach().cpu().numpy()); l_.extend(y.cpu().numpy())
        sg_sched.step()
        if ep % 5 == 0 or ep == 1:
            tr = accuracy_score(l_, p_)
            _, sv, _, _, _, _ = evaluate(sg_model, src_val_loader, sg_crit, DEVICE)
            _, tv, _, _, _, _ = evaluate(sg_model, tgt_val_loader, sg_crit, DEVICE)
            print(f"  Ep {ep:2d}/30 | TRAIN acc={tr:.4f} | SRC-VAL acc={sv:.4f} | TGT-VAL acc={tv:.4f}")

    _, sv, sf, sp, sl, _ = evaluate(sg_model, src_val_loader, sg_crit, DEVICE)
    _, tv, tf, tp, tl, _ = evaluate(sg_model, tgt_val_loader, sg_crit, DEVICE)

    print(f"\n  FINAL  SRC-VAL acc = {sv:.4f}   f1 = {sf:.4f}")
    print(f"         TGT-VAL acc = {tv:.4f}   f1 = {tf:.4f}  (source-only baseline, no DA)")
    print("\n  Source-val confusion matrix (rows = actual):")
    cm = confusion_matrix(sl, sp)
    print("  " + " " * 26 + "".join(f"{c[:13]:>15s}" for c in CLASSES))
    for i, c in enumerate(CLASSES):
        print(f"  {c:<26s}" + "".join(f"{cm[i][j]:>15d}" for j in range(NUM_CLASSES)))

    print("\n  " + "-" * 60)
    if sv >= 0.97:
        print("  VERDICT: PASS. Architecture and optimizer are healthy.")
        print("           v6's 0.730 plateau came from the recipe (pseudo-label poisoning,")
        print("           entropy collapse, KD on blurry inputs) -- all addressed below.")
    elif sv >= 0.90:
        print("  VERDICT: MARGINAL. Not collapse, but under-fitting. Try LR 2e-3 / 50 epochs")
        print("           before moving on, and check the augmentation preview in Part 1E.")
    else:
        print("  VERDICT: FAIL. A class is still being erased WITHOUT any DA loss.")
        print("           Do not proceed. Check the confusion matrix above: if one row is")
        print("           empty, inspect the class folders and the augmentation preview.")
    print("  " + "-" * 60)

    sanity_src_val, sanity_tgt_val = sv, tv
    del sg_model, sg_opt
    torch.cuda.empty_cache()
else:
    print("RUN_SANITY_GATE=False -- skipped. Run it at least once on a fresh environment.")
    sanity_src_val = sanity_tgt_val = None

  SANITY GATE: source-only training. Target >= 0.97 source-val accuracy.
  Ep  1/30 | TRAIN acc=0.5918 | SRC-VAL acc=0.2624 | TGT-VAL acc=0.3671
  Ep  5/30 | TRAIN acc=0.7704 | SRC-VAL acc=0.9078 | TGT-VAL acc=0.4430
  Ep 10/30 | TRAIN acc=0.8534 | SRC-VAL acc=0.9468 | TGT-VAL acc=0.5063
  Ep 15/30 | TRAIN acc=0.8719 | SRC-VAL acc=0.9504 | TGT-VAL acc=0.5696
  Ep 20/30 | TRAIN acc=0.9035 | SRC-VAL acc=0.9681 | TGT-VAL acc=0.5570
  Ep 25/30 | TRAIN acc=0.9267 | SRC-VAL acc=0.9929 | TGT-VAL acc=0.5570
  Ep 30/30 | TRAIN acc=0.9217 | SRC-VAL acc=0.9858 | TGT-VAL acc=0.5696

  FINAL  SRC-VAL acc = 0.9858   f1 = 0.9858
         TGT-VAL acc = 0.5696   f1 = 0.5467  (source-only baseline, no DA)

  Source-val confusion matrix (rows = actual):
                              Tomato_Septor  Potato_Early_  Corn_Leaf_Bli
  Tomato_Septoria_Leaf_Spot             266              0              0
  Potato_Early_Blight                     7            143              0
  Corn_Leaf_Blight               

---
# Part 5: Teacher

The teacher is the accuracy ceiling for everything downstream, so it is fixed first. It is
fine-tuned with the **same single DA technique under test in this notebook** (CDAN), so
its soft targets already reflect that technique rather than some different combined recipe.

- **Pseudo-labeling off by default.** Establish the clean number first. Only then consider
  `TEACHER_USE_PSEUDO=True`, which goes through the EMA + class-balanced + auto-off path.
- **Discriminative learning rates.** Low backbone LR (3e-4) with a high head LR (3e-3) plus
  warmup, so the randomly initialised head converges without dragging the pretrained features.
- **Patience 30, 80 epochs**, with warmup + cosine.
- **Same strong augmentation** as the student, at native 224px.
- For the adversarial techniques (DANN/CDAN) a small `domain_clf` sized to the teacher's own
  `feature_dim` is trained alongside it and added to the optimizer's parameter groups; it is
  discarded after training, same as `kd_proj`.

Watch `TRAIN` alongside `SRC-VAL`: source saturating near 0.99 in the first epochs is expected
and is precisely why the augmentation and the target-side alignment term have to carry the run.


In [ ]:
# ============================================================
# PART 5: TEACHER FINE-TUNING
# ============================================================
if USE_TEACHER_KD:
    set_seed(SEED)
    teacher = TeacherNet().to(DEVICE)
    t_params = sum(p.numel() for p in teacher.parameters())
    print(f"Teacher: {TEACHER_BACKBONE}, {t_params:,} params (training-time only, never exported)")
    print(f"DA technique under test: {DA_METHOD.upper()}")

    # discriminative LR instead of a cosmetic block freeze
    head_ids = {id(p) for p in teacher.classifier.parameters()}
    t_domain_clf = (make_domain_clf(teacher.feature_dim, DA_METHOD).to(DEVICE)
                    if DA_METHOD in ('dann', 'cdan') else None)
    t_param_groups = [
        {'params': [p for p in teacher.parameters() if id(p) not in head_ids],
         'lr': TEACHER_LR_BACKBONE},
        {'params': list(teacher.classifier.parameters()), 'lr': TEACHER_LR_HEAD},
    ]
    if t_domain_clf is not None:
        t_param_groups.append({'params': list(t_domain_clf.parameters()), 'lr': TEACHER_LR_HEAD})
    t_opt = optim.AdamW(t_param_groups, weight_decay=WEIGHT_DECAY)
    t_sched = make_scheduler(t_opt, TEACHER_EPOCHS, TEACHER_WARMUP)
    t_stop = EarlyStopping(patience=TEACHER_PATIENCE, min_delta=1e-3)
    t_ema = EMA(teacher) if TEACHER_USE_PSEUDO else None

    t_hist = {'src_val': [], 'tgt_val': [], 'train': [], DA_METHOD: []}
    t_raw, t_best_raw, t_best_sm, t_best_state = [], 0.0, 0.0, copy.deepcopy(teacher.state_dict())
    t_pseudo_on, t_last_ps_acc = False, float('nan')
    t_plateau = 0
    t0 = time.time()

    print("=" * 96)
    print(f"  Teacher | {TEACHER_EPOCHS} epochs | 224px | DA={DA_METHOD.upper()} | pseudo-labeling="
          f"{'ON' if TEACHER_USE_PSEUDO else 'OFF (establish the clean number first)'}")
    print("=" * 96)

    for ep in range(1, TEACHER_EPOCHS + 1):
        st = train_da_epoch(teacher, src_train_loader, tgt_train_loader, cls_criterion,
                            t_opt, DEVICE, ep, TEACHER_EPOCHS, CFG, which='t',
                            teacher=None, kd_proj=None, ema=t_ema,
                            pseudo_enabled=t_pseudo_on, use_kd=False,
                            da_method=DA_METHOD, domain_clf=t_domain_clf)
        t_sched.step()

        _, sv, _, _, _, _ = evaluate(teacher, src_val_loader, cls_criterion, DEVICE, 't')
        _, tv, tf1, _, _, _ = evaluate(teacher, tgt_val_loader, cls_criterion, DEVICE, 't')

        t_raw.append(tv)
        sm = float(np.mean(t_raw[-SMOOTH_WINDOW:]))
        t_hist['src_val'].append(sv); t_hist['tgt_val'].append(tv)
        t_hist['train'].append(st['train_acc']); t_hist[DA_METHOD].append(st[f'{DA_METHOD}_loss'])

        t_best_raw = max(t_best_raw, tv)
        if sm > t_best_sm:
            t_best_sm = sm
            t_best_state = copy.deepcopy(teacher.state_dict())
        t_stop(sm, teacher)

        # ---- pseudo-label control: plateau gate + hard auto-off ----
        if TEACHER_USE_PSEUDO:
            if len(t_raw) > SMOOTH_WINDOW and sm <= t_best_sm:
                t_plateau += 1
            else:
                t_plateau = 0
            if (not t_pseudo_on) and t_plateau >= PSEUDO_PLATEAU_EPOCHS:
                t_pseudo_on = True
                print(f"  >> pseudo-labeling ENABLED at epoch {ep} (target-val plateaued)")
            if st['pseudo_used'] > 0:
                t_last_ps_acc = st['pseudo_acc']
                if t_last_ps_acc < PSEUDO_MIN_MONITOR_ACC:
                    t_pseudo_on = False
                    print(f"  >> pseudo-labeling AUTO-DISABLED: monitor acc "
                          f"{t_last_ps_acc:.3f} < {PSEUDO_MIN_MONITOR_ACC}")

        if ep % 5 == 0 or ep == 1:
            ps = f"{st['pseudo_acc']:.3f}" if st['pseudo_acc'] == st['pseudo_acc'] else "n/a"
            print(f"[T] Ep {ep:3d}/{TEACHER_EPOCHS} | cls={st['cls_loss']:.4f} "
                  f"{DA_METHOD}={st[f'{DA_METHOD}_loss']:.4f} | "
                  f"TRAIN={st['train_acc']:.3f} SRC-VAL={sv:.3f} | TGT-VAL={tv:.3f} "
                  f"(sm={sm:.3f}) f1={tf1:.3f} | pseudo#={st['pseudo_used']} acc={ps} "
                  f"| {time.time()-t0:.0f}s")

        if t_stop.early_stop:
            print(f"[T] Early stop at epoch {ep}")
            break

    teacher.load_state_dict(t_best_state)
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    torch.save(teacher.state_dict(), f"{MODELS_DIR}/teacher.pth")

    print(f"\nTeacher done in {time.time()-t0:.0f}s")
    print(f"  best RAW target-val      : {t_best_raw:.4f}")
    print(f"  best SMOOTHED target-val : {t_best_sm:.4f}")
    print("  This is the realistic ceiling for the student -- KD cannot exceed it.")
else:
    teacher, t_best_sm = None, 0.0
    print("USE_TEACHER_KD=False -- student will train with DA only.")


Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 128MB/s] 


Teacher: mobilenet_v3_large, 2,974,835 params (training-time only, never exported)
DA technique under test: CDAN
  Teacher | 80 epochs | 224px | DA=CDAN | pseudo-labeling=OFF (establish the clean number first)
[T] Ep   1/80 | cls=0.4923 cdan=0.2072 | TRAIN=0.900 SRC-VAL=0.995 | TGT-VAL=0.696 (sm=0.696) f1=0.699 | pseudo#=0 acc=n/a | 35s
[T] Ep   5/80 | cls=0.3288 cdan=0.2587 | TRAIN=0.990 SRC-VAL=0.998 | TGT-VAL=0.747 (sm=0.714) f1=0.747 | pseudo#=0 acc=n/a | 172s
[T] Ep  10/80 | cls=0.3235 cdan=0.3040 | TRAIN=0.991 SRC-VAL=0.972 | TGT-VAL=0.734 (sm=0.737) f1=0.727 | pseudo#=0 acc=n/a | 342s
[T] Ep  15/80 | cls=0.3328 cdan=0.2895 | TRAIN=0.988 SRC-VAL=0.998 | TGT-VAL=0.722 (sm=0.734) f1=0.719 | pseudo#=0 acc=n/a | 513s
[T] Ep  20/80 | cls=0.3207 cdan=0.2968 | TRAIN=0.991 SRC-VAL=0.995 | TGT-VAL=0.747 (sm=0.752) f1=0.747 | pseudo#=0 acc=n/a | 684s
[T] Ep  25/80 | cls=0.3282 cdan=0.3262 | TRAIN=0.990 SRC-VAL=0.996 | TGT-VAL=0.747 (sm=0.744) f1=0.743 | pseudo#=0 acc=n/a | 856s
[T] Ep  30/

In [ ]:
# ============================================================
# PART 5B: TEACHER SANITY + AdaBN CHECK
# FIX #13: AdaBN keep/discard is now decided on target-VAL, never target-TEST.
# The old version compared t_ab_acc vs t_te_acc (both computed on tgt_test_loader) and
# kept whichever scored higher ON THE TEST SET -- that's test-set peeking: reporting the
# best-of-2 test score instead of a single clean held-out measurement. Now the decision
# uses tgt_val_loader, and tgt_test_loader is touched exactly once, with the model the
# decision already settled on.
# ============================================================
if USE_TEACHER_KD:
    _, t_val_acc, _, _, _, _ = evaluate(teacher, tgt_val_loader, cls_criterion, DEVICE, 't')

    teacher_adabn = copy.deepcopy(teacher)
    adapt_bn(teacher_adabn, tgt_train_loader, DEVICE, which='t', n_pass=3)
    _, t_ab_val_acc, _, _, _, _ = evaluate(teacher_adabn, tgt_val_loader, cls_criterion, DEVICE, 't')

    print(f"Teacher target-VAL (no AdaBN) : acc={t_val_acc:.4f}")
    print(f"Teacher target-VAL (+ AdaBN)  : acc={t_ab_val_acc:.4f} (delta {t_ab_val_acc - t_val_acc:+.4f})")

    if t_ab_val_acc >= t_val_acc:
        teacher = teacher_adabn
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        print("  -> AdaBN version selected (decided on target-VAL, not test).")
    else:
        del teacher_adabn
        print("  -> Base (non-AdaBN) version kept (decided on target-VAL, not test).")
    torch.cuda.empty_cache()

    # Test set touched exactly once, with whichever model the VAL-based decision above settled on.
    _, t_te_acc, t_te_f1, t_pred, t_lab, _ = evaluate(teacher, tgt_test_loader,
                                                      cls_criterion, DEVICE, 't')
    print(f"\nTeacher target-TEST (final, single measurement) : acc={t_te_acc:.4f} f1={t_te_f1:.4f}")
    print("Teacher target-test confusion (rows = actual):")
    cm = confusion_matrix(t_lab, t_pred)
    print(" " * 27 + "".join(f"{c[:13]:>15s}" for c in CLASSES))
    for i, c in enumerate(CLASSES):
        print(f"{c:<27s}" + "".join(f"{cm[i][j]:>15d}" for j in range(NUM_CLASSES)))


Teacher target-VAL (no AdaBN) : acc=0.7342
Teacher target-VAL (+ AdaBN)  : acc=0.7595 (delta +0.0253)
  -> AdaBN version selected (decided on target-VAL, not test).

Teacher target-TEST (final, single measurement) : acc=0.6709 f1=0.6790
Teacher target-test confusion (rows = actual):
                             Tomato_Septor  Potato_Early_  Corn_Leaf_Bli
Tomato_Septoria_Leaf_Spot               22              2              0
Potato_Early_Blight                     14             12              0
Corn_Leaf_Blight                        10              0             19


---
# Part 5C: Student

The deployed model: distilled from the teacher, adapted with **CDAN only**, optionally
self-trained. This is the notebook's single controlled variable versus the other six baselines:

- KD weight is **1.0 and constant**, and the teacher is fed its own paired 224px view of the
  same augmented image rather than a 96→224 upsample.
- **Feature distillation** through `kd_proj` supplements logit KD, which matters at a ~70x
  capacity gap. `kd_proj` is trained alongside the student and is not part of the exported graph.
- Alignment uses **CDAN only** — every other technique's lambda in `CFG` is left at zero
  for this run (see `train_da_epoch`'s `use_*` flags, all derived from `DA_METHOD`).
- Source and target share **one forward pass** for clean BN statistics.
- Pseudo-labeling waits for a target-val plateau, draws labels from the EMA model on the weak
  view, selects class-balanced top-k, and **switches itself off** below 0.85 monitor accuracy.
- For DANN/CDAN, `st_domain_clf` is created here, sized to the student's `feature_dim`, and its
  parameters are added to the student's optimizer alongside `kd_proj`.

`COLLAPSE WARNING` fires whenever source-val lands within half a point of any `1 - class_share`
value. If that appears, stop the run — something is erasing a class regardless of the DA method.


In [ ]:
# ============================================================
# PART 5C: STUDENT TRAINING
# ============================================================

def train_student(seed=SEED, epochs=EPOCHS, verbose=True, tag="student"):
    set_seed(seed)
    st_model = MicroBrainNetDA().to(DEVICE)
    kd_proj = (nn.Linear(st_model.feature_dim, teacher.feature_dim).to(DEVICE)
               if (USE_TEACHER_KD and teacher is not None) else None)
    st_domain_clf = (make_domain_clf(st_model.feature_dim, DA_METHOD).to(DEVICE)
                     if DA_METHOD in ('dann', 'cdan') else None)

    params = (list(st_model.parameters())
             + (list(kd_proj.parameters()) if kd_proj else [])
             + (list(st_domain_clf.parameters()) if st_domain_clf else []))
    opt = optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    sched = make_scheduler(opt, epochs, WARMUP_EPOCHS)
    stop = EarlyStopping(patience=PATIENCE, min_delta=1e-3)
    ema = EMA(st_model) if STUDENT_USE_PSEUDO else None

    hist = {k: [] for k in ['cls', DA_METHOD, 'pseudo', 'kd', 'featkd',
                            'train_acc', 'src_val', 'tgt_val', 'tgt_f1', 'lam', 'lr',
                            'pseudo_used', 'pseudo_acc']}
    raw, best_raw, best_sm = [], 0.0, 0.0
    best_state = copy.deepcopy(st_model.state_dict())
    pseudo_on, plateau, collapse_warned = False, 0, False

    shares = np.array([src_index.labels.count(i) for i in range(NUM_CLASSES)], float)
    shares /= shares.sum()
    collapse_vals = [1 - s for s in shares]
    t0 = time.time()

    if verbose:
        print("=" * 100)
        print(f"  Student [{tag}] seed={seed} | {epochs} epochs | DA={DA_METHOD.upper()} | KD="
              f"{'ON (lambda=%.1f const)' % LAMBDA_KD if kd_proj else 'OFF'} | "
              f"DA steps/epoch={min(len(src_train_loader), DA_TARGET_PASSES*len(tgt_train_loader))}")
        print("=" * 100)

    for ep in range(1, epochs + 1):
        st = train_da_epoch(st_model, src_train_loader, tgt_train_loader, cls_criterion,
                            opt, DEVICE, ep, epochs, CFG, which='s',
                            teacher=teacher if USE_TEACHER_KD else None,
                            kd_proj=kd_proj, ema=ema, pseudo_enabled=pseudo_on,
                            da_method=DA_METHOD, domain_clf=st_domain_clf)
        sched.step()

        _, sv, _, _, _, _ = evaluate(st_model, src_val_loader, cls_criterion, DEVICE)
        _, tv, tf1, _, _, _ = evaluate(st_model, tgt_val_loader, cls_criterion, DEVICE)

        raw.append(tv)
        sm = float(np.mean(raw[-SMOOTH_WINDOW:]))
        for k, v in [('cls', st['cls_loss']), (DA_METHOD, st[f'{DA_METHOD}_loss']),
                     ('pseudo', st['pseudo_loss']), ('kd', st['kd_loss']), ('featkd', st['featkd_loss']),
                     ('train_acc', st['train_acc']), ('src_val', sv), ('tgt_val', tv),
                     ('tgt_f1', tf1), ('lam', st['lam']), ('lr', sched.get_last_lr()[0]),
                     ('pseudo_used', st['pseudo_used']), ('pseudo_acc', st['pseudo_acc'])]:
            hist[k].append(v)

        best_raw = max(best_raw, tv)
        if sm > best_sm:
            best_sm = sm
            best_state = copy.deepcopy(st_model.state_dict())
        stop(sm, st_model)

        # ---- collapse detector: "one class absorbed" signature ----
        if (not collapse_warned) and ep > 12 and any(abs(sv - c) < 0.005 for c in collapse_vals):
            collapse_warned = True
            print(f"  !! COLLAPSE WARNING ep {ep}: src-val {sv:.4f} matches a "
                  f"'one class absorbed' value. Stop and inspect.")

        # ---- pseudo-label control ----
        if STUDENT_USE_PSEUDO:
            plateau = plateau + 1 if (len(raw) > SMOOTH_WINDOW and sm <= best_sm) else 0
            if (not pseudo_on) and plateau >= PSEUDO_PLATEAU_EPOCHS:
                pseudo_on = True
                if verbose:
                    print(f"  >> pseudo-labeling ENABLED at epoch {ep}")
            if st['pseudo_used'] > 0 and st['pseudo_acc'] < PSEUDO_MIN_MONITOR_ACC:
                pseudo_on = False
                if verbose:
                    print(f"  >> pseudo-labeling AUTO-DISABLED (monitor acc "
                          f"{st['pseudo_acc']:.3f} < {PSEUDO_MIN_MONITOR_ACC})")

        if verbose and (ep % 5 == 0 or ep == 1):
            ps = f"{st['pseudo_acc']:.3f}" if st['pseudo_acc'] == st['pseudo_acc'] else "n/a"
            print(f"Ep {ep:3d}/{epochs} | cls={st['cls_loss']:.3f} "
                  f"{DA_METHOD}={st[f'{DA_METHOD}_loss']:.4f} "
                  f"kd={st['kd_loss']:.3f} fkd={st['featkd_loss']:.3f} | "
                  f"TRAIN={st['train_acc']:.3f} SRC-VAL={sv:.3f} | TGT-VAL={tv:.3f} "
                  f"(sm={sm:.3f}) f1={tf1:.3f} | ps#={st['pseudo_used']} acc={ps} "
                  f"| {time.time()-t0:.0f}s")

        if stop.early_stop:
            if verbose:
                print(f"\nEarly stop at epoch {ep}")
            break

    st_model.load_state_dict(best_state)
    if verbose:
        print(f"\nDone in {time.time()-t0:.0f}s | best RAW tgt-val {best_raw:.4f} | "
              f"best SMOOTHED (selected) {best_sm:.4f}")
    return st_model, kd_proj, hist, best_sm


model, kd_proj, history, best_smoothed = train_student()
torch.save({'model_state_dict': model.state_dict(), 'history': history,
            'config': {'img_size': IMG_SIZE, 'channels': STUDENT_CHANNELS,
                       'classes': CLASSES, 'params': total_params,
                       'method': f'{DA_METHOD.upper()} (single-technique baseline) '
                                 f'+ gatedPseudo + KD(logit+feature)'}},
           f"{MODELS_DIR}/student_final.pth")
print(f"Saved -> {MODELS_DIR}/student_final.pth")


  Student [student] seed=42 | 120 epochs | DA=CDAN | KD=ON (lambda=1.0 const) | DA steps/epoch=24
Ep   1/120 | cls=0.957 cdan=0.3027 kd=1.108 fkd=0.002 | TRAIN=0.551 SRC-VAL=0.264 | TGT-VAL=0.380 (sm=0.380) f1=0.223 | ps#=0 acc=n/a | 33s
Ep   5/120 | cls=0.700 cdan=0.2765 kd=0.753 fkd=0.001 | TRAIN=0.746 SRC-VAL=0.926 | TGT-VAL=0.443 (sm=0.423) f1=0.435 | ps#=0 acc=n/a | 163s
Ep  10/120 | cls=0.572 cdan=0.2657 kd=0.561 fkd=0.001 | TRAIN=0.840 SRC-VAL=0.952 | TGT-VAL=0.494 (sm=0.522) f1=0.507 | ps#=0 acc=n/a | 326s
  >> pseudo-labeling ENABLED at epoch 13
Ep  15/120 | cls=0.523 cdan=0.2784 kd=0.490 fkd=0.001 | TRAIN=0.866 SRC-VAL=0.972 | TGT-VAL=0.608 (sm=0.572) f1=0.614 | ps#=0 acc=n/a | 487s
Ep  20/120 | cls=0.489 cdan=0.2847 kd=0.431 fkd=0.001 | TRAIN=0.886 SRC-VAL=0.993 | TGT-VAL=0.658 (sm=0.635) f1=0.657 | ps#=0 acc=n/a | 650s
Ep  25/120 | cls=0.437 cdan=0.2751 kd=0.353 fkd=0.001 | TRAIN=0.918 SRC-VAL=0.984 | TGT-VAL=0.671 (sm=0.651) f1=0.661 | ps#=0 acc=n/a | 818s
Ep  30/120 | cls

---
# Part 6: AdaBN + evaluation

`adapt_bn` runs on a copy of the student using target-train images only, then both versions are
scored on the target test set and the better one is kept. This must happen **before** INT8
calibration in Part 10, otherwise quantization freezes the wrong normalization statistics into
the exported graph.

Every number is reported with a bootstrap 95% confidence interval. On 79 test images a single
image is worth 1.27 points and the interval is roughly ±9 points wide — two runs whose accuracies
fall inside each other's interval are not meaningfully different, and reporting a bare point
estimate would be misleading.

In [ ]:
# ============================================================
# PART 6: AdaBN + FULL EVALUATION
# ============================================================

def bootstrap_ci(labels, preds, n_boot=5000, seed=SEED):
    rng = np.random.RandomState(seed)
    labels, preds = np.asarray(labels), np.asarray(preds)
    n = len(labels)
    accs = np.array([(labels[i] == preds[i]).mean()
                     for i in (rng.randint(0, n, n) for _ in range(n_boot))])
    return accs.mean(), *np.percentile(accs, [2.5, 97.5])


# FIX #13: AdaBN keep/discard decided on target-VAL, never target-TEST (see Part 5B for
# why the old test-vs-test comparison was test-set peeking).
_, base_val_acc, _, _, _, _ = evaluate(model, tgt_val_loader, cls_criterion, DEVICE)

model_ab = copy.deepcopy(model)
adapt_bn(model_ab, tgt_train_loader, DEVICE, which='s', n_pass=3)
_, ab_val_acc, _, _, _, _ = evaluate(model_ab, tgt_val_loader, cls_criterion, DEVICE)

print(f"Student target-VAL (no AdaBN) : acc={base_val_acc:.4f}")
print(f"Student target-VAL (+ AdaBN)  : acc={ab_val_acc:.4f} (delta {ab_val_acc-base_val_acc:+.4f})")

if ab_val_acc >= base_val_acc:
    model = model_ab
    print("  -> AdaBN version selected (decided on target-VAL, not test).")
else:
    del model_ab
    print("  -> Base (non-AdaBN) version kept (decided on target-VAL, not test).")

# Test set touched exactly once, with whichever model the VAL-based decision above settled on.
_, tgt_acc, tgt_f1, tgt_preds, tgt_labels, tgt_probs = evaluate(model, tgt_test_loader, cls_criterion, DEVICE)
print(f"\nStudent target-TEST (final, single measurement) : acc={tgt_acc:.4f} f1={tgt_f1:.4f}")
torch.save(model.state_dict(), f"{MODELS_DIR}/student_deploy.pth")

# FIX #12: clean (non-augmented) source-train loader and correct per-split predictions
_, src_train_acc, src_train_f1, sp_tr, sl_tr, _ = evaluate(model, src_train_eval_loader,
                                                           cls_criterion, DEVICE)
_, src_acc, src_f1, src_preds, src_labels, src_probs = evaluate(model, src_val_loader,
                                                                cls_criterion, DEVICE)
_, src_test_acc, src_test_f1, sp_te, sl_te, _ = evaluate(model, src_test_loader,
                                                         cls_criterion, DEVICE)

results = OrderedDict()
for name, (a, f, p, l) in [('Source Train (clean)', (src_train_acc, src_train_f1, sp_tr, sl_tr)),
                           ('Source Val',           (src_acc, src_f1, src_preds, src_labels)),
                           ('Source Test',          (src_test_acc, src_test_f1, sp_te, sl_te)),
                           ('Target Test',          (tgt_acc, tgt_f1, tgt_preds, tgt_labels))]:
    results[name] = {'accuracy': a, 'f1': f,
                     'precision': precision_score(l, p, average='weighted', zero_division=0),
                     'recall': recall_score(l, p, average='weighted', zero_division=0)}

print(f"\n{'='*78}")
print(f"{'Split':<24}{'Accuracy':<12}{'F1':<12}{'Precision':<12}{'Recall':<12}")
print("-" * 78)
for k, v in results.items():
    print(f"{k:<24}{v['accuracy']:<12.4f}{v['f1']:<12.4f}"
          f"{v['precision']:<12.4f}{v['recall']:<12.4f}")
print("=" * 78)
pd.DataFrame(results).T.to_csv(f"{TABLES_DIR}/performance_summary.csv")

bm, lo, hi = bootstrap_ci(tgt_labels, tgt_preds)
print(f"\nTarget test accuracy : {tgt_acc:.4f}")
print(f"Bootstrap 95% CI     : [{lo:.4f}, {hi:.4f}]  (n={len(tgt_labels)}, "
      f"1 image = {100/len(tgt_labels):.2f} points)")
print("Report the interval, not the point estimate.")

print(f"\nGeneralization gap (clean src-train -> tgt-test): {src_train_acc - tgt_acc:.4f}")
if USE_TEACHER_KD:
    print(f"\nCapacity chain on target test:")
    print(f"  Teacher ({TEACHER_BACKBONE}, 224px, {t_params/1e6:.1f}M) : {t_te_acc:.4f}")
    print(f"  Student ({IMG_SIZE}px, {total_params/1e3:.0f}K, DEPLOYED) : {tgt_acc:.4f}")
    print(f"  KD retention: {tgt_acc/max(t_te_acc,1e-9)*100:.1f}% of the teacher")

print("\nTarget-test classification report:")
print(classification_report(tgt_labels, tgt_preds, target_names=CLASSES, digits=4))

Student target-VAL (no AdaBN) : acc=0.7468
Student target-VAL (+ AdaBN)  : acc=0.6582 (delta -0.0886)
  -> Base (non-AdaBN) version kept (decided on target-VAL, not test).

Student target-TEST (final, single measurement) : acc=0.6203 f1=0.6105

Split                   Accuracy    F1          Precision   Recall      
------------------------------------------------------------------------------
Source Train (clean)    0.9977      0.9977      0.9977      0.9977      
Source Val              0.9982      0.9982      0.9982      0.9982      
Source Test             0.9982      0.9982      0.9982      0.9982      
Target Test             0.6203      0.6105      0.6993      0.6203      

Target test accuracy : 0.6203
Bootstrap 95% CI     : [0.5190, 0.7215]  (n=79, 1 image = 1.27 points)
Report the interval, not the point estimate.

Generalization gap (clean src-train -> tgt-test): 0.3775

Capacity chain on target test:
  Teacher (mobilenet_v3_large, 224px, 3.0M) : 0.6709
  Student (128px, 167